In [1]:
import pickle, gzip
import numpy as np

import awkward as ak
import numpy as np

import matplotlib.pyplot as plt
import mplhep as hep

import hist
from hist import Hist

from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
from topcoffea.modules.histEFT import HistEFT
NanoAODSchema.warn_missing_crossrefs = False

from coffea.analysis_tools import PackedSelection
from topcoffea.modules import utils
import topcoffea.modules.eft_helper as efth

import re
import pickle, gzip

import os
import cloudpickle

# Testing PDF variation calculation

In [36]:
# hists = pickle.load(gzip.open("Run2_ttbarEFT.pkl.gz"))
hists = pickle.load(gzip.open("sow_MC_2017.pkl.gz"))
hists = pickle.load(gzip.open("./TT01j2l_SM_sow.pkl.gz"))

PDFweights = hists['sow_LHEPDFweights'][{'process':'UL17_TTGJets_ptSkim'}]

updates = {}
for i in PDFweights.axes['PDFindex']:
    SOW = PDFweights[{'PDFindex':i}].values()[0]
    # print(f"index {i}: \t {SOW}")
    updates[f"nSumOfWeights_LHEPDFweights{i}"] = SOW

In [21]:
h_view = PDFweights[{'process':sum, 'channel':sum,}].values()
nominal = h_view[:, 0]
variations = h_view[:, 1:]

diff_sq = np.square(variations - nominal[:, np.newaxis])
sigma_pdf = np.sqrt(np.sum(diff_sq, axis=1))

pdf_up = nominal + sigma_pdf
pdf_down = nominal - sigma_pdf

print(nominal)
print(pdf_up)
print(pdf_down)

[ 0.          0.          0.          0.          0.          0.
  0.          0.         -0.09930324  0.04463808  0.31467024  0.63975826
 -0.58615635  0.08664807  0.81064954  0.12879129  1.71841763  1.08670843
  1.98491242  1.5524464   1.57238921  0.63069755  1.27268502  0.5472916
  0.50827342  1.08016846  0.78424562  0.16304831  0.26146598  0.34126722
 -0.03532402  1.13505048  0.75368295  0.19404783  0.19333106  0.58871129
  0.16089782 -0.00765787  0.57411402  0.05847268  0.10243509  0.42914156
  0.2104308   0.18746677  0.09050364  0.18165086  0.09891231  0.23082382
  0.20054261  0.08836291]
[ 0.          0.          0.          0.          0.          0.
  0.          0.         -0.09085111  0.04749913  0.33196625  0.68003757
 -0.5021788   0.11325765  0.86895125  0.16169997  1.81580281  1.29073235
  2.10748359  1.66258309  1.6491778   0.67646176  1.36747997  0.62342402
  0.53688361  1.20710135  0.84284415  0.17165402  0.27165683  0.43103647
 -0.02471512  1.21270388  0.83774926  0.21

# Why are some events failing after the PDF sow normalization is added? 
There are some events in this file that has a zero length for the LHEPdfWeight so then the numpy broadcast fails. 

In [8]:
fname = "/cms/cephfs/data//store/user/hnelson2/mc/ttbarEFT_Run2/UL18/postSIM/v4/NAOD_TTto2L2Nu_1Jets_smeft_MTT_0to700/NAOD-00000_111707.root"

events1 = NanoEventsFactory.from_root(
    {fname: "Events"},
    schemaclass=NanoAODSchema,
    metadata={"dataset": "TTto2L2Nu"},
    mode="eager",
).events()

norm_ratios = np.array([
    1.2
    for i in range(len(events1.LHEPdfWeight[0])) # Assumes consistent length
])

print(events1.LHEPdfWeight)
# print(np.shape(events1.LHEPdfWeight))

weights = events1.LHEPdfWeight
lengths = ak.num(weights)
counts = np.unique(ak.to_numpy(lengths), return_counts=True)
print(f"Occurrences of lengths: {dict(zip(counts[0], counts[1]))}")

if len(counts[0]) > 1:
    print("Found inconsistent lengths!")
    # Find the indices of the 'bad' events
    standard_length = counts[0][np.argmax(counts[1])]
    bad_event_indices = np.where(ak.to_numpy(lengths) != standard_length)[0]
    print(f"First 10 bad event indices: {bad_event_indices[:50]}")

[[1, 0.999, 0.999, 1, 1, 1, 0.999, 1, ..., 1, 1, 1, 1, 1, 1, 0.968, 1.04], ...]
Occurrences of lengths: {np.int64(0): np.int64(360), np.int64(103): np.int64(71416)}
Found inconsistent lengths!
First 10 bad event indices: [5352 5353 5354 5355 5356 5357 5358 5359 5360 5361 5362 5363 5364 5365
 5366 5367 5368 5369 5370 5371 5372 5373 5374 5375 5376 5377 5378 5379
 5380 5381 5382 5383 5384 5385 5386 5387 5388 5389 5390 5391 5392 5393
 5394 5395 5396 5397 5398 5399 5400 5401]


In [2]:
fname = "/cms/cephfs/data//store/user/hnelson2/mc/ttbarEFT_Run2/UL18/postSIM/v4/NAOD_TTto2L2Nu_1Jets_smeft_MTT_0to700/NAOD-00000_295759.root"

events = NanoEventsFactory.from_root(
    {fname: "Events"},
    schemaclass=NanoAODSchema,
    metadata={"dataset": "TTto2L2Nu"},
    mode="eager",
    # entry_stop=10000,
).events()

norm_ratios = np.array([
    1.2
    for i in range(len(events.LHEPdfWeight[0])) # Assumes consistent length
])

print(events.LHEPdfWeight)
print(np.shape(events.LHEPdfWeight))
pdf_weights_unmasked = events.LHEPdfWeight * norm_ratios[np.newaxis, :]

print(ak.num(events.LHEPdfWeight) == 103)

[[1, 1, 0.998, 1, 1, 0.993, 1, 0.998, ..., 1, 1, 1, 1, 1, 1, 0.958, 1.05], ...]
[10000, 103]
[True, True, True, True, True, True, ..., True, True, True, True, True, True]


I ran over the ttbar EFT samples and summed up how many events in each dataset had bad events. 
It's only present in 2017 and 2018 and the % there is very low. 

In [23]:
h = pickle.load(gzip.open("./PDFweight_check.pkl.gz"))
for p in h['sow'].keys():
    hist = h['sow'][p]['event_quality']
    print(f"{p}")
    print(f"\t total events: \t {hist[{'type':'total'}].values()}")
    print(f"\t bad pdf events: \t {hist[{'type':'bad_pdf'}].values()}")
    print(f"\t % bad events: \t {(hist[{'type':'bad_pdf'}].values()/hist[{'type':'total'}].values())[0]*100} %")

UL16APV_TT01j2l_mtt0to700
	 total events: 	 [12951196.]
	 bad pdf events: 	 [0.]
	 % bad events: 	 0.0 %
UL16APV_TT01j2l_mtt700to900
	 total events: 	 [1691404.]
	 bad pdf events: 	 [0.]
	 % bad events: 	 0.0 %
UL16APV_TT01j2l_mtt900toInf
	 total events: 	 [2781037.]
	 bad pdf events: 	 [0.]
	 % bad events: 	 0.0 %
UL16_TT01j2l_mtt0to700
	 total events: 	 [7252593.]
	 bad pdf events: 	 [0.]
	 % bad events: 	 0.0 %
UL16_TT01j2l_mtt700to900
	 total events: 	 [2023970.]
	 bad pdf events: 	 [0.]
	 % bad events: 	 0.0 %
UL16_TT01j2l_mtt900toInf
	 total events: 	 [2979128.]
	 bad pdf events: 	 [0.]
	 % bad events: 	 0.0 %
UL17_TT01j2l_mtt0to700
	 total events: 	 [13573075.]
	 bad pdf events: 	 [8579.]
	 % bad events: 	 0.06320601632275664 %
UL17_TT01j2l_mtt700to900
	 total events: 	 [3768792.]
	 bad pdf events: 	 [4275.]
	 % bad events: 	 0.11343157170785759 %
UL17_TT01j2l_mtt900toInf
	 total events: 	 [6111671.]
	 bad pdf events: 	 [10899.]
	 % bad events: 	 0.17833093437130368 %
UL18_TT01j

# Testing Fix

In [6]:
fname = "file:///cms/cephfs/data//store/user/hnelson2/mc/ttbarEFT_Run2/UL17/postLHE/v3/NAOD_TTto2L2Nu_1Jets_smeft_MTT_0to700_v1/NAOD-00000_1119686.root"
events = NanoEventsFactory.from_root(
    {fname: "Events"},
    schemaclass=NanoAODSchema,
    metadata={"dataset": "TTto2L2Nu"},
    mode="eager",
).events()

In [7]:
pdf_mask = (ak.num(events.LHEPdfWeight) == 103)
pdf_mask

<Array [True, True, True, True, ..., True, True, True, True] type='383 * bool'>

In [9]:
norm_ratios = np.array([
    1.2 for i in range(103)
])

In [13]:
current_pdf_weights = ak.to_regular(events.LHEPdfWeight[pdf_mask])
pdf_weights_unmasked = current_pdf_weights * norm_ratios

# Some problems with the tW weights

In [ ]:
fname = "/project01/ndcms/hnelson2//store/user/hnelson2/skims/mc/ptSkim/tW_NoFullyHadronicDecays/UL16_TW_antitop_5f_NoFullyHadronicDecays/output_124.root"

events = NanoEventsFactory.from_root(
    {fname: "Events"},
    schemaclass=NanoAODSchema,
    metadata={"dataset": "tW"},
    mode="eager",
    # entry_stop=10000,
).events()

KeyboardInterrupt: 

In [ ]:
ak.sum(events.LHEPdfWeight[0])

np.float32(102.80487)

In [12]:
events.LHEPdfWeight[0]

<Array [0.998, 0.997, 0.996, ..., 0.961, 1.04] type='103 * float32[paramete...'>

In [2]:
fname = "/users/hnelson2/09D7D088-C170-9A45-97F4-66EBDD4F9361.root"
events_orig = NanoEventsFactory.from_root(
    {fname: "Events"},
    schemaclass=NanoAODSchema,
    metadata={"dataset": "tW"},
    mode="eager",
    entry_stop=10000,
).events()

In [3]:
events_orig.LHEPdfWeight[0]

<Array [0.141, 0.141, 0.139, ..., 0.139, 0.14] type='101 * float32[paramete...'>

In [5]:
ak.sum(events_orig.LHEPdfWeight[0])

np.float32(14.20826)

In [6]:
fname = "/users/hnelson2/09D7D088-C170-9A45-97F4-66EBDD4F9361.root"
events = NanoEventsFactory.from_root(
    {fname: "Events"},
    schemaclass=NanoAODSchema,
    metadata={"dataset": "tW"},
    mode="eager",
).events()

In [9]:
ak.sum(events.LHEPdfWeight[0])

np.float32(14.20826)

In [10]:
events["genWeight"]

<Array [32.5, 32.5, 32.5, ..., 32.5, 32.5] type='45342 * float32[parameters...'>

In [20]:
ak.sum(events.LHEPdfWeight, axis=0)

<Array [-2.9e+03, -2.85e+03, ..., -2.91e+03, -2.92e+03] type='101 * float32'>

In [24]:
ak.sum(events.LHEPdfWeight[3])

np.float32(-3.85507)

In [31]:
len(events.LHEPdfWeight[98])

np.float32(102.60266)

In [44]:
ak.num(events.LHEPdfWeight)

<Array [101, 101, 101, 101, 101, ..., 101, 101, 101, 101] type='45342 * int64'>

In [46]:
for i in range(ak.num(events.LHEPdfWeight)[0]):
    # print(i)
    print(f"index {i}: \t sum={ak.sum(events.LHEPdfWeight[i])}")

index 0: 	 sum=14.208259582519531
index 1: 	 sum=-27.940399169921875
index 2: 	 sum=-0.50368332862854
index 3: 	 sum=-3.855070114135742
index 4: 	 sum=0.0
index 5: 	 sum=-10.521480560302734
index 6: 	 sum=-12.683509826660156
index 7: 	 sum=-7.9849090576171875
index 8: 	 sum=75.07864379882812
index 9: 	 sum=-11.9913330078125
index 10: 	 sum=-11.215606689453125
index 11: 	 sum=-20.548057556152344
index 12: 	 sum=-17.838409423828125
index 13: 	 sum=-30.629653930664062
index 14: 	 sum=-0.39196932315826416
index 15: 	 sum=103.00723266601562
index 16: 	 sum=-29.226898193359375
index 17: 	 sum=-9.796550750732422
index 18: 	 sum=-6.190898895263672
index 19: 	 sum=-35.18171691894531
index 20: 	 sum=-26.62689208984375
index 21: 	 sum=-13.403533935546875
index 22: 	 sum=-12.487224578857422
index 23: 	 sum=0.0
index 24: 	 sum=-7.132144927978516
index 25: 	 sum=24.154090881347656
index 26: 	 sum=21.807151794433594
index 27: 	 sum=-6.905239105224609
index 28: 	 sum=-9.740962982177734
index 29: 	 sum